# Neuron FRI: Response & Polarity Analysis

This notebook simulates neural responses to **ON** and **OFF** stimuli using the **SIC model** to derive the **Full-Range Index (FRI)**, a metric for quantifying functional polarity (ON- vs. OFF-preference).

### Workflow:
* **Stimuli Synthesis**: Generates step-function tensors to trigger ON/OFF dynamics.
* **Response Inference**: Predicts baseline-corrected activation traces ($R_{on}, R_{off}$) using GPU-accelerated simulation.
* **Data Preparation**: Exports response features for subsequent FRI calculation and neuron classification.

## 1. Environment Setup and Dependency Loading
This block configures the system path to include local utility modules and imports the core SIC (Signal Infinite Cascade) model components.

In [1]:
import sys
import os

sys.path.append(os.path.abspath('../util'))

from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch

## 2. Neural Response Simulation (FRI Pipeline)
This module generates standardized ON/OFF visual stimuli and executes the **SIC model** to simulate neural activity, providing the raw response traces ($R_{on}$ and $R_{off}$) required to compute the **Full-Range Index (FRI)**.

### Operational Workflow:
- **Stimuli Generation**: Synthesizes 3D step-function tensors ($41 \times 82 \times 2000$) for ON and OFF transitions.
- **Model Execution**: Runs GPU-accelerated simulations to derive temporal response dynamics for the target neuron population.
- **Data Export**: Processes responses with baseline correction, outputting files to `../results/responses/fri/` for downstream FRI polarity analysis.

In [ ]:
import numpy as np
from tqdm import tqdm
from datetime import datetime
import csv
import torch

# --- 1. Synthetic Stimuli Construction for FRI Calculation ---
print("Generating two custom stimuli...")

stimulus_dataset = {}

# ON-step: Dark to Light transition for R_on measurement
ON = np.zeros((41, 82, 2000), dtype=np.float32)
ON[:, :, 1000:] = 1.0
stimulus_dataset["ON"] = ON

# OFF-step: Light to Dark transition for R_off measurement
OFF = np.ones((41, 82, 2000), dtype=np.float32)
OFF[:, :, 1000:] = 0.0
stimulus_dataset["OFF"] = OFF

print(f"✅ Total stimuli: {len(stimulus_dataset)}")

# --- 2. SIC Model Initialization ---
SIC_model = SICModelTorch(
    matrix_dir="neuron_matrices",
    output_dir="../results/responses/fri",
    t_step=10,
    rate=100,
    device=torch.device("cuda:0")
)

# --- 3. Data Loading Utilities ---
def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")
    return neuron_ids

# Extract target neuron list and load their corresponding weights
neuron_ids = read_all_neuron_ids()
print(f"Found {len(neuron_ids)} neurons")

# Pre-load weight matrices into memory (without further normalization)
weights = SIC_model.load_weights(neuron_ids, normalize=False)
print("Weights loaded successfully")

# --- 4. SIC Response Calculation Loop ---
print("Calculating SIC responses...")
for stim_name, stimulus in tqdm(
    stimulus_dataset.items(),
    desc="Processing stimuli",
    unit="stimulus"
):
    SIC_model.calculate_response_baseline(
        stimulus,
        weights,
        responce_threshold=0,
        baseline_steps=50,
        stim_name=stim_name
    )

# --- 5. Execution Summary ---
print("\n" + "="*50)
print("PREPROCESSING COMPLETE")
print("="*50)
print(f"Total stimuli processed: {len(stimulus_dataset)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)


Generating two custom stimuli...
✅ Total stimuli: 2
Using device: cuda:1
Found 139255 neurons
Weights loaded successfully
Calculating SIC responses...


Processing stimuli: 100%|██████████| 2/2 [00:23<00:00, 11.69s/stimulus]


PREPROCESSING COMPLETE
Total stimuli processed: 2
Timestamp: 2026-03-10 17:13:23
